In [42]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ucphm import *
from functions.ct_column import *
from scipy.stats import skew, shapiro

In [43]:
dataset = pd.read_excel('../../../BreastCareCond.xlsx')
data_code = pd.read_excel('../../../Codificação variaveis.xlsx')

Using the variables she used

In [ ]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB']
categorical_variables = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient', 
                         'Side location of the lesion', 'Classification with respect to other lesions', 
                         'Source of referral', 'Type of invasion at CB', 'Istotype at CB', 'Grade at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Her2 overexpression (with immunohystochemistry) at CB', 'Disease extent', 
                         'Ultrasound scan', 'Pathological size (mm.)', 'Tumour size by imaging or clinical examination']
date_columns = ['Date of CB ', 'Date of 1st breast operation', 'Date of intervention on sentinel lymph node',
                'Date of intervention (axilla)']
selected_variables = relevant_variables + date_columns+ numerical_variables + categorical_variables

Replace missing values first

In [45]:
missing_values(dataset, categorical_variables)

,Column,Missing Values,Percentage Missing
0,N (Regional nodes affected),173,4.469129
1,Number identifying the lesion in the patient,0,0.000000
2,Side location of the lesion,20,0.516662
3,Classification with respect to other lesions,12,0.309997
4,Source of referral,0,0.000000
5,Type of invasion at CB,324,8.369930
6,Istotype at CB,257,6.639111
7,Grade at CB,387,9.997417
8,Oestrogen receptor status at CB,314,8.111599
9,Progesterone receptor status at CB,691,17.850685


In [27]:
replace_zero = ['N (Regional nodes affected)', 'Type of invasion at CB', 'Grade at CB', 
                'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Her2 overexpression (with immunohystochemistry) at CB', 
                'Ultrasound scan', 'Pathological size (mm.)', 'Tumour size by imaging or clinical examination']
replace_one = ['Classification with respect to other lesions']
replace_four = ['Istotype at CB', 'Disease extent']
replace_missing_values(dataset, replace_zero, 0)
replace_missing_values(dataset, replace_one, 1)
replace_missing_values(dataset, replace_four, 4)
# replace the missing values with the mode of the column
mode_value = dataset['Side location of the lesion'].mode()[0]
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].fillna(mode_value)

Done
Done
Done


Creating the cT variable like she did

In [28]:
grouped_T(dataset)
cT (dataset)
clinical_staging (dataset)
new_variables = ['cT']
selected_variables = selected_variables + new_variables
categorical_variables = categorical_variables + new_variables
dataset['cT'] = dataset['cT'].replace('TX', np.nan)

In [29]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [30]:
cT = ['cT']
replace_missing_values(dataset, cT, 'unknown')

Done


To reduce the number of values

In [31]:
simplify_2 = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient']
simplify_3 = ['Istotype at CB']
simplify_variable(dataset, simplify_2, 2)
simplify_variable(dataset, simplify_3, 3)

Done
Done


In [32]:
test_normality(dataset, numerical_variables)

,Variable,Skewness,Shapiro p-value,Normality,Has Outliers
0,Age,0.213822,9.449767e-17,Probably not normal,False
1,Ki67 positive cells (%) at CB,0.973698,5.684308e-41,Probably not normal,True


Use the median for the tree based models, it is only implemented later because I want to export it to ucphm_diagnostic

To do label encoding

In [33]:
dataset['Number identifying the lesion in the patient'] = dataset['Number identifying the lesion in the patient'].map({'1' : 0, '>1' : 1})
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].map({'L' : 0,'R' : 1})
dataset['Source of referral'] = dataset['Source of referral'].map({1 : 0, 2 : 1})
dataset['N (Regional nodes affected)'] = dataset['N (Regional nodes affected)'].map({'0.0' : 0, '1.0' : 1, '>1' : 2})
dataset['Istotype at CB'] = dataset['Istotype at CB'].map({'1.0' : 1, '2.0' : 2, '>2' : 3})

In [34]:
pie_chart(dataset, categorical_variables)

'Saved.'

In [35]:
one_hot_variables = ['N (Regional nodes affected)', 'Classification with respect to other lesions', 'Type of invasion at CB', 'Istotype at CB', 'Grade at CB', 
                     'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Her2 overexpression (with immunohystochemistry) at CB', 
                     'Disease extent', 'cT', 'Ultrasound scan']
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables)

In [36]:
ohe_dataset = date_of_diagnosis(ohe_dataset, date_columns)

In [37]:
selected_variables = [col for col in selected_variables if col not in one_hot_variables]
selected_variables = [col for col in selected_variables if col not in date_columns]
selected_variables.extend(new_column_names.tolist())
date_of_diagnosis = ['Date of diagnosis']
selected_variables = selected_variables + date_of_diagnosis

categorical_variables = [col for col in categorical_variables if col not in one_hot_variables]
categorical_variables.extend(new_column_names.tolist())
survival_dataset = ohe_dataset[selected_variables].copy()

In [38]:
with open("numerical_ultrasound.txt", "w") as file:
    for item in numerical_variables:
        file.write(str(item) + "\n")
with open("categorical_ultrasound.txt", "w") as file:
    for item in categorical_variables:
        file.write(str(item) + "\n")

In [39]:
relevant_dates = ['Date of diagnosis', 'Date of 1st regional recurrence']
survival_dataset = remove_zeros_na(survival_dataset, relevant_dates)
survival_dataset = format_date(survival_dataset, relevant_dates)

In [40]:
survival_dataset = survival_days(survival_dataset, ['Date of diagnosis'], ['Date of 1st regional recurrence'])
survival_dataset = tte_negative_patients(survival_dataset)
survival_dataset = filter_EBC(survival_dataset)
survival_dataset.reset_index(drop=True, inplace=True)

In [41]:
survival_dataset.to_csv("data_treated_ultrasound.csv")

The nan values of the numerical data are not filled